In [8]:
from pathlib import Path
import sys

# Add project root directory
sys.path.append(str(Path().resolve().parent))

from src.constants.paths import SECRET_PATH
from src.processing.pde_ple import PDE, PLE, es
from src.processing.document import Document
from src.processing.utils import clean_text

from src.constants.constants import (
    USEFUL_EXTENSIONS,
    ENVIRONNEMENT_PATH,
    S3_RAW_DOCS_PATH,
    SERVICES_PATH,
    ENVIRONNEMENT_RAW_PATH,
)

from elasticsearch.helpers import scan, bulk
from elasticsearch import Elasticsearch

# ----------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------

source_index = "sg2-constat-entrepot"
target_index = "uc202-rex-cameleon"
scroll_window = "10m"
batch_size = 5000

# ----------------------------------------------------------
# CREATE TARGET INDEX (with nested + face_name mapping)
# ----------------------------------------------------------

if not es.indices.exists(index=target_index):
    mapping = {
        "mappings": {
            "dynamic": True,
            "properties": {
                "face_list": {
                    "type": "nested",
                    "properties": {
                        # we explicitly create face_name to allow exact + analyzed search
                        "face_name": {
                            "type": "keyword",
                            "ignore_above": 256,
                            "fields": {"text": {"type": "text", "analyzer": "french"}},
                        }
                    },
                }
            },
        }
    }
    es.indices.create(index=target_index, body=mapping)

# ----------------------------------------------------------
# FIELDS TO EXTRACT AND CONCATENATE
# ----------------------------------------------------------
fields_to_extract = [
    "coeur_reference_text",
    "coeur_unite_creatrice_text",
    "coeur_localisation_keyword",
    "coeur_date_constat_date",
    "coeur_nature_text",
    "coeur_occasion_decouverte_keyword",
    "coeur_titre_text",
    "coeur_description_text",
    "coeur_action_proposition_text",
    "coeur_action_immediate_text",
    "face_list.liste_nom_faces_text",
    "face_list.face_unite_responsable_text",
    "face_list.face_field_etat_text",
    "face_list.dipnn_rex_doe_pilote_dinstruction_et_appui___technical_project_lead_and_support_correspondant_user",
    "face_list.dipnn_rex_doe_DESCRIPTION_DETAILLEE_text",
    "face_list.dipnn_rex_doe_PROJET_SOURCE_text",
    "face_list.dipnn_rex_doe_projets_concernes_text",
    "face_list.dipnn_rex_doe_SYSTEME_text",
    "face_list.dipnn_rex_doe_SECTEUR_BAT_text",
    "face_list.dipnn_rex_doe_DOMAINE_text",
    "face_list.dipnn_rex_doe_CONTRAT_text",
    "face_list.dipnn_rex_doe_COMMENTAIRES_IMPACTS_text",
    "face_list.dipnn_rex_doe_ENJEU_text",
    "face_list.dipnn_rex_doe_ANALYSE_CAUSES_PROFONDES_text",
    "face_list.dipnn_rex_doe_SOLUTIONS_CORRECTIVES_text",
    "face_list.dipnn_rex_doe_ENSEIGNEMENTS_APPRIS_text",
    "face_list.dipnn_rex_doe_PROCESSUS_CONCERNES_ENSEIGNEMENTS_RECOMMANDATIONS_text",
    "face_list.dipnn_rex_doe_REFERENTIELS_TECHNIQUES_INGENIERIE_CONCERNES_ENSEIGNEMENTS_RECOMMANDATIONS_text",
    "face_list.dipnn_rex_doe_PRODUCT_BREAKDOWN_STRUCURE_text",
    "face_list.dipnn_rex_doe_APPLICABILITE_text",
    "face_list.dipnn_rex_doe_COMMENTAIRE_COMITE_VALIDATION_text",
    "face_list.dipnn_rex_doe_references_des_actions_creees_text",
    "face_list.dipnn_rex_doe_REFERENCE_ECM_text",
    "face_list.dipnn_rex_doe_cree_par_user",
    "face_list.dipnn_petal_CATEGORIE_text",
    "face_list.dipnn_petal_SYSTEMES_ELEMENTAIRES_text",
    "face_list.dipnn_petal_REFERENCES_text",
    "face_list.dipnn_petal_PROJETS_CONCERNES_text",
    "face_list.dipnn_petal_APPLICABILITE_PARC_text",
    "face_list.dipnn_petal_ANALYSE_IMPACTS_AVANT_TRAITEMENT_text",
    "face_list.dipnn_petal_impacts_documentaires_text",
    "face_list.dipnn_petal_REFERENTIEL_CONCERNÉ_text",
    "face_list.dipnn_petal_COMMENTAIRES_text",
    "face_list.dipnn_petal_TYPE_AIP_text",
    "face_list.dipnn_petal_FAMILLE_AIP_text",
    "face_list.dipnn_petal_EXIGENCE_DEFINIE_NON_RESPECTEE_POUR_AIP_text",
    "face_list.dipnn_petal_EIP_CONCERNE_text",
    "face_list.dipnn_petal_EXIGENCE_DEFINIE_NON_RESPECTEE_EIP_text",
    "face_list.dipnn_petal_ANALYSE_CAUSES_text",
    "face_list.dipnn_petal_CAUSE_PRINCIPALE_text",
    "face_list.dipnn_petal_PROCESSUS_SMI_CONCERNE_text",
    "face_list.dipnn_petal_MODE_TRAITEMENT_DECIDE_text",
    "face_list.dipnn_petal_CODE_TECHNIQUE_DEROGE_text",
    "face_list.dipnn_petal_COMMENTAIRES_text",
]

# Fields to concatenate
fields_to_concatenate = [
    "coeur_reference_text",
    "coeur_unite_creatrice_text",
    "coeur_localisation_keyword",
    # "coeur_date_constat_date",
    # "coeur_nature_text",
    "coeur_occasion_decouverte_keyword",
    "coeur_titre_text",
    "coeur_description_text",
    "coeur_action_proposition_text",
    "coeur_action_immediate_text",
    # "face_list.liste_nom_faces_text",
    # "face_list.face_unite_responsable_text",
    # "face_list.dipnn_rex_doe_REX_BOUCLE_COURTE_text",
    "face_list.dipnn_rex_doe_pilote_dinstruction_et_appui___technical_project_lead_and_support_correspondant_user",
    "face_list.dipnn_rex_doe_DESCRIPTION_DETAILLEE_text",
    "face_list.dipnn_rex_doe_PROJET_SOURCE_text",
    "face_list.dipnn_rex_doe_SYSTEME_text",
    "face_list.dipnn_rex_doe_SECTEUR_BAT_text",
    "face_list.dipnn_rex_doe_DOMAINE_text",
    "face_list.dipnn_rex_doe_CONTRAT_text",
    "face_list.dipnn_rex_doe_COMMENTAIRES_IMPACTS_text",
    "face_list.dipnn_rex_doe_ENJEU_text",
    "face_list.dipnn_rex_doe_ANALYSE_CAUSES_PROFONDES_text",
    "face_list.dipnn_rex_doe_SOLUTIONS_CORRECTIVES_text",
    "face_list.dipnn_rex_doe_ENSEIGNEMENTS_APPRIS_text",
    "face_list.dipnn_rex_doe_PROCESSUS_CONCERNES_ENSEIGNEMENTS_RECOMMANDATIONS_text",
    "face_list.dipnn_rex_doe_REFERENTIELS_TECHNIQUES_INGENIERIE_CONCERNES_ENSEIGNEMENTS_RECOMMANDATIONS_text",
    "face_list.dipnn_rex_doe_PRODUCT_BREAKDOWN_STRUCURE_text",
    "face_list.dipnn_rex_doe_APPLICABILITE_text",
    "face_list.dipnn_rex_doe_COMMENTAIRE_COMITE_VALIDATION_text",
    "face_list.dipnn_rex_doe_references_des_actions_creees_text",
    # "face_list.dipnn_rex_doe_REFERENCE_ECM_text",  #rajouter ##sniffdoc et joindre
    "face_list.dipnn_rex_doe_cree_par_user",
    "face_list.dipnn_petal_CATEGORIE_text",
    "face_list.dipnn_petal_SYSTEMES_ELEMENTAIRES_text",
    "face_list.dipnn_petal_REFERENCES_text",
    "face_list.dipnn_petal_PROJETS_CONCERNES_text",
    "face_list.dipnn_petal_APPLICABILITE_PARC_text",
    "face_list.dipnn_petal_ANALYSE_IMPACTS_AVANT_TRAITEMENT_text",
    "face_list.dipnn_petal_impacts_documentaires_text",
    "face_list.dipnn_petal_REFERENTIEL_CONCERNÉ_text",
    "face_list.dipnn_petal_COMMENTAIRES_text",
    "face_list.dipnn_petal_TYPE_AIP_text",
    "face_list.dipnn_petal_FAMILLE_AIP_text",
    "face_list.dipnn_petal_EXIGENCE_DEFINIE_NON_RESPECTEE_POUR_AIP_text",
    "face_list.dipnn_petal_EIP_CONCERNE_text",
    "face_list.dipnn_petal_EXIGENCE_DEFINIE_NON_RESPECTEE_EIP_text",
    "face_list.dipnn_petal_ANALYSE_CAUSES_text",
    "face_list.dipnn_petal_CAUSE_PRINCIPALE_text",
    "face_list.dipnn_petal_PROCESSUS_SMI_CONCERNE_text",
    "face_list.dipnn_petal_MODE_TRAITEMENT_DECIDE_text",
    "face_list.dipnn_petal_CODE_TECHNIQUE_DEROGE_text",
    "face_list.dipnn_petal_COMMENTAIRES_text",
]
# ----------------------------------------------------------
# QUERY FILTER
# ----------------------------------------------------------

query = {
    "query": {
        "bool": {
            "must": [
                {
                    "nested": {
                        "path": "face_list",
                        "query": {
                            "bool": {
                                "should": [
                                    {
                                        "wildcard": {
                                            "face_list.liste_nom_faces_text": "*REX Ingénierie*"
                                        }
                                    },
                                    {
                                        "wildcard": {
                                            "face_list.liste_nom_faces_text": "*Constat Écart Ingénierie*"
                                        }
                                    },
                                ],
                                "minimum_should_match": 1,
                            }
                        },
                    }
                }
            ]
        }
    }
}

# ----------------------------------------------------------
# EXECUTE SEARCH
# ----------------------------------------------------------

documents = scan(
    es, index=source_index, query=query, scroll=scroll_window, size=batch_size
)

actions = []

# ----------------------------------------------------------
# UTILS
# ----------------------------------------------------------


def split_face_names(raw):
    """
    Split concatenated face names into a clean list.
    Handles strings (comma-separated) or lists.
    """
    if raw is None:
        return []
    if isinstance(raw, list):
        # already a list of names
        return [clean_text(str(x)).strip() for x in raw if x not in (None, "")]
    if isinstance(raw, str):
        # split on commas, strip whitespace
        parts = [clean_text(p).strip() for p in raw.split(",")]
        # Remove empties
        return [p for p in parts if p]
    # fallback
    return [clean_text(str(raw)).strip()]


# ----------------------------------------------------------
# MAIN LOOP – rebuild nested face_list with exploded face_name
# ----------------------------------------------------------

for doc in documents:
    source = doc.get("_source", {})
    new_doc = {}
    content_parts = []

    # -------------------------
    # 1) FLAT FIELDS
    # -------------------------
    for field in fields_to_extract:
        if field.startswith("face_list."):
            continue
        value = source.get(field, None)
        if value is not None:
            new_doc[field] = value
            # add to content
            content_parts.append(str(value))

    # -------------------------
    # 2) NESTED face_list + explode liste_nom_faces_text
    # -------------------------
    nested_items = []

    src_faces = source.get("face_list")
    if isinstance(src_faces, list) and src_faces:
        # Derive the list of face names from the FIRST face's liste_nom_faces_text,
        # which appears identical across all in your sample.
        first_raw_names = src_faces[0].get("liste_nom_faces_text")
        names = split_face_names(first_raw_names)

        for idx, face in enumerate(src_faces):
            new_face = {}

            # Assign per-face name by index if available; otherwise fallback:
            face_name = names[idx] if idx < len(names) else None

            # If no indexed name available, try splitting the current face’s own value:
            if face_name in (None, ""):
                face_name = split_face_names(face.get("liste_nom_faces_text"))[:1]
                face_name = face_name[0] if face_name else None

            if face_name:
                new_face["face_name"] = face_name
                # also add to content
                content_parts.append(face_name)

            # Copy other nested fields
            for f in fields_to_extract:
                if not f.startswith("face_list."):
                    continue
                subfield = f.split(".", 1)[1]  # e.g., "face_unite_responsable_text"
                # we skip liste_nom_faces_text to avoid re-introducing concatenated noise,
                # but if you want to keep it, uncomment next 2 lines:
                # if subfield == "liste_nom_faces_text":
                #     continue
                val = face.get(subfield)
                if val not in (None, "", []):
                    new_face[subfield] = val
                    # For content, we add *some* nested fields, but avoid re-adding
                    # the long concatenated names:
                    if subfield != "liste_nom_faces_text":
                        content_parts.append(str(val))


# Filter out unwanted face_field_etat_text values
            banned_states = {"Annulé", "6. Annulée"}

            etat = new_face.get("face_field_etat_text")

            if etat not in banned_states:
                nested_items.append(new_face)

    # Keep only the faces you want
    allowed_faces = {"Constat Écart Ingénierie", "REX Ingénierie"}

    filtered_items = [f for f in nested_items if f.get("face_name") in allowed_faces]

    if filtered_items:
        new_doc["face_list"] = filtered_items
    else:
        new_doc["face_list"] = []

    # -------------------------
    # 3) CONCATENATED CONTENT
    # -------------------------
    new_doc["content"] = clean_text(" ".join([str(x) for x in content_parts]))
    new_doc["source"] = "cameleon"

    actions.append({"_index": target_index, "_source": new_doc})

    if len(actions) >= batch_size:
        bulk(es, actions)
        actions = []

# Final bulk
if actions:
    bulk(es, actions)

print(
    "Migration terminée avec succès vers l'index 'uc202-rex-cameleon' (face_name explosé)."
)


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/tmp/ipykernel_12021/3473038054.py:55: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es.indices.create(index=target_index, body=mapping)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py

Migration terminée avec succès vers l'index 'uc202-rex-cameleon' (face_name explosé).


In [4]:
from elasticsearch import Elasticsearch

# Quick sanity check: does the query match anything?
resp_count = es.count(index=source_index, body=query)
print(f"[COUNT] matched docs: {resp_count.get('count')}")

# Fetch a small sample to inspect fields and values
resp_search = es.search(
    index=source_index,
    body={
        **query,
        "_source": ["face_list.liste_nom_faces_text"],  # add more fields if helpful
        "size": 5,
        "track_total_hits": True,
    },
)
total = resp_search.get("hits", {}).get("total", {})
print(
    f"[SEARCH] total: {total} | sample hits: {len(resp_search.get('hits', {}).get('hits', []))}"
)
for i, h in enumerate(resp_search.get("hits", {}).get("hits", []), 1):
    print(f"--- hit {i} ---")
    print(h.get("_id"))
    print(h.get("_source", {}))


/tmp/ipykernel_45962/3319196241.py:4: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  resp_count = es.count(index=source_index, body=query)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/tmp/ipykernel_45962/3319196241.py:8: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  resp_search = es.search(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly 

[COUNT] matched docs: 0
[SEARCH] total: {'value': 0, 'relation': 'eq'} | sample hits: 0


In [3]:
import boto3


# Initialiser le client S3
s3 = boto3.client(
    "s3",
    region_name="eu-west-1",
    endpoint_url= "https://s3-interne-dsit.edf.fr",
    aws_access_key_id="20E3TDRLTLI06Z8UW3B7",
    aws_secret_access_key="4mhkP7mjrPeCmKgN7UIPgD8aBs5OfixOTY5nFB6z",
    verify=False,  # Désactive la vérification SSL
)


# Specify the bucket name and the key (path) where you want to upload the file
bucket_name = "bkt-pud-uc"
object_key = "uc202-rex/test.md"  # Replace 'your_file_name.parquet' with the desired S3 object key

# Path to the local file you want to upload
file_path = (
    "/opt/app-root/src/uc202-ipn-rex/README.md"  # Replace with the path to your local file
)

# Upload the file
s3.upload_file(file_path, bucket_name, object_key)

print(f"File uploaded to {bucket_name}/{object_key}")


File uploaded to bkt-pud-uc/uc202-rex/test.md


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 's3-interne-dsit.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [13]:
from elasticsearch import Elasticsearch

# Initialize the Elasticsearch client

# Index name
index_name = "uc202-rex-cameleon"

# Reference text to search for
reference_text = "C0000227685"

# Query to find the document with the specific coeur_reference_text
query = {
    "query": {"bool": {"must": [{"match": {"coeur_reference_text": reference_text}}]}}
}

# Perform the search
response = es.search(index=index_name, body=query)

# Extract the content attribute
if response["hits"]["total"]["value"] > 0:
    # Assuming there's only one document with the given coeur_reference_text
    document = response["hits"]["hits"][0]["_source"]
    content_value = document.get("content", "No content found")
    print(f"Content: {content_value}")
else:
    print("No document found with the given coeur_reference_text.")


Content: C0000227685 DISC EDVANCE Montrouge - Park Azur Constat spontané REX de l'outil conduite L’outil CONDUITE est une application informatique d’édition permettant d’élaborer les Pièces 6 (P6) des Dossiers de Système Elémentaire sur les projets FA3 et HPC du palier EPR. Ces P6 se décomposent en 3 : 
-	Les P6.1, images de supervision 
-	Les P6.2, fiches d’alarme 
-	Les P6.3, Modes OPératoires. 

L'objectif de ce REX sur l'outil CONDUITE est de pouvoir anticiper, pour les projets futurs, l’organisation et les problématiques que nous pouvions rencontrées durant les activités de production.
 [{'nni': 'C68948', 'nom': 'NEROT', 'prenom': 'THOMAS', 'fullName': 'Thomas NEROT'}] 

Gestion de configuration des données 


L’outils CONDUITE FA3 ne fait pas de la gestion de configuration pour ses données, pour une version N des P6 on ne peut plus avoir une version N-1, N-2, ... de celles-ci.
On ne peut donc pas avoir plusieurs versions de P6 en simultanée dans une même base de données. 


Dans 

/tmp/ipykernel_26616/1598334552.py:17: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = es.search(index=index_name, body=query)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
clean_text(content_value)


'C0000227685 DISC EDVANCE Montrouge - Park Azur Constat spontané REX de l\'outil conduite L’outil CONDUITE est une application informatique d’édition permettant d’élaborer les Pièces 6 (P6) des Dossiers de Système Elémentaire sur les projets FA3 et HPC du palier EPR. Ces P6 se décomposent en 3 : \n-\tLes P6.1, images de supervision \n-\tLes P6.2, fiches d’alarme \n-\tLes P6.3, Modes OPératoires. \n\nL\'objectif de ce REX sur l\'outil CONDUITE est de pouvoir anticiper, pour les projets futurs, l’organisation et les problématiques que nous pouvions rencontrées durant les activités de production.\n [{\'nni\': \'C68948\', \'nom\': \'NEROT\', \'prenom\': \'THOMAS\', \'fullName\': \'Thomas NEROT\'}] \n\nGestion de configuration des données \n\n\nL’outils CONDUITE FA3 ne fait pas de la gestion de configuration pour ses données, pour une version N des P6 on ne peut plus avoir une version N-1, N-2, ... de celles-ci.\nOn ne peut donc pas avoir plusieurs versions de P6 en simultanée dans une même

In [12]:
from pathlib import Path
import sys

# Add the project root directory to sys.path
sys.path.append(str(Path().resolve().parent))
from src.constants.paths import SECRET_PATH


from src.processing.pde_ple import PDE, PLE, es
from src.processing.document import Document

# get all paths
from src.constants.constants import (
    USEFUL_EXTENSIONS,
    ENVIRONNEMENT_PATH,
    S3_RAW_DOCS_PATH,
    SERVICES_PATH,
    ENVIRONNEMENT_RAW_PATH,
)
